# Gemini vs DeepSeek Code Comparison

This notebook compares code differences between source and mutant files using Google's Gemini and DeepSeek AI models.

## Features
- Batch processing of multiple source/mutant pairs
- Results stored in pandas DataFrame
- Robust API key handling and error management
- Cleaned code comparison (comments removed)


In [ ]:
# Install required dependencies
%pip install google-generativeai requests pandas matplotlib seaborn python-dotenv

print("✅ Dependencies installation complete!")


In [ ]:
# Import necessary libraries
import os
import re
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from typing import Dict, List, Any, Tuple, Optional
from dotenv import load_dotenv
import google.generativeai as genai
import requests
import warnings
warnings.filterwarnings('ignore')

# Load environment variables from .env file (if it exists)
load_dotenv()

print("✅ Libraries imported successfully!")


## API Configuration (Secure)

**🔒 Security Best Practices:**

API keys are loaded securely using one of these methods (in order of preference):

1. **`.env` file** (Recommended for local development): 
   - Copy `env_template.txt` to `.env` in this directory
   - Fill in your API keys in the `.env` file
   - The `.env` file is automatically ignored by git (see `.gitignore`)

2. **Environment Variables** (Recommended for production): Set in your shell
   ```bash
   export LLM_API_KEY='your_key_here'
   export LLM_API_KEY='your_key_here'
   ```

3. **Jupyter Secure Input**: For one-time use (see optional cell below)
   - Uses `getpass` for hidden input
   - Keys stored only in memory for the session

**⚠️ NEVER commit API keys to version control!**


### Option: Secure Input via Jupyter (One-time session)

If you prefer to enter keys interactively (keys are hidden and only stored in memory for the session):


In [ ]:
# Optional: Interactive secure input (only use if keys not in .env or environment)
# This prompts for keys securely (input is hidden) - keys are only stored in memory
# Run this cell AFTER cell 6, or if you want to enter keys interactively instead

from getpass import getpass

# Check if variables exist, if not load from environment
if 'LLM_API_KEY' not in globals():
    LLM_API_KEY = os.getenv('LLM_API_KEY', '').strip()

if 'LLM_API_KEY' not in globals():
    LLM_API_KEY = os.getenv('LLM_API_KEY', '').strip()

# Only prompt if keys aren't already set
if not LLM_API_KEY:
    LLM_API_KEY = getpass('Enter Gemini API Key (hidden): ').strip()
    if LLM_API_KEY:
        os.environ['LLM_API_KEY'] = LLM_API_KEY
        try:
            genai.configure(api_key=LLM_API_KEY)
            gemini_model = genai.GenerativeModel('gemini-gemini-2.5-pro')
            print("✅ Gemini API configured (interactive input)")
        except Exception as e:
            print(f"⚠️  Error: {e}")
            gemini_model = None

if not LLM_API_KEY:
    LLM_API_KEY = getpass('Enter DeepSeek API Key (hidden): ').strip()
    if LLM_API_KEY:
        os.environ['LLM_API_KEY'] = LLM_API_KEY
        deepseek_headers = {
            "Authorization": f"Bearer {LLM_API_KEY}",
            "Content-Type": "application/json"
        }
        print("✅ DeepSeek API configured (interactive input)")
    else:
        deepseek_headers = None
else:
    # Ensure deepseek_headers is set if key exists
    if LLM_API_KEY and ('deepseek_headers' not in globals() or deepseek_headers is None):
        deepseek_headers = {
            "Authorization": f"Bearer {LLM_API_KEY}",
            "Content-Type": "application/json"
        }

# Re-check configuration
if 'gemini_model' in globals() and gemini_model is not None:
    print("\n✅ Gemini API ready")
if 'deepseek_headers' in globals() and deepseek_headers is not None:
    print("✅ DeepSeek API ready")
    
if ('gemini_model' not in globals() or gemini_model is None) and \
   ('deepseek_headers' not in globals() or deepseek_headers is None):
    print("\n⚠️  Note: Run cell 6 first to load keys from .env/environment, or enter keys above")


In [ ]:
# API Configuration - Load from environment variables or .env file
# Keys are loaded securely via load_dotenv() above or system environment variables

LLM_API_KEY = os.getenv('LLM_API_KEY', '').strip()
LLM_API_KEY = os.getenv('LLM_API_KEY', '').strip()

# Validate and configure Gemini
gemini_model = None
if LLM_API_KEY:
    try:
        genai.configure(api_key=LLM_API_KEY)
        gemini_model = genai.GenerativeModel('gemini-3.1-pro-preview')
        print("✅ Gemini API configured successfully (loaded from environment/.env)")
    except Exception as e:
        print(f"⚠️  Error configuring Gemini API: {e}")
        gemini_model = None
else:
    print("⚠️  Gemini API key not found.")
    print("   💡 Set LLM_API_KEY in environment or .env file")

# DeepSeek API configuration
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"
deepseek_headers = None

if LLM_API_KEY:
    deepseek_headers = {
        "Authorization": f"Bearer {LLM_API_KEY}",
        "Content-Type": "application/json"
    }
    print("✅ DeepSeek API configured successfully (loaded from environment/.env)")
else:
    print("⚠️  DeepSeek API key not found.")
    print("   💡 Set LLM_API_KEY in environment or .env file")

# Check if at least one API is configured
if gemini_model is None and deepseek_headers is None:
    print("\n❌ ERROR: No APIs configured!")
    print("\n📝 Setup Instructions:")
    print("   1. Create a .env file in this directory with:")
    print("      LLM_API_KEY=your_key_here")
    print("      LLM_API_KEY=your_key_here")
    print("   2. OR set environment variables:")
    print("      export LLM_API_KEY='your_key_here'")
    print("      export LLM_API_KEY='your_key_here'")
    print("\n   🔒 Make sure .env is in .gitignore!")
else:
    print("\n✅ Ready to run comparisons")


## File Reading and Prompt Generation

Functions to read and clean C source files, and generate prompts for comparison.


In [ ]:
# System and User Prompts
SYSTEM_PROMPT = (
    "You are a helpful and precise coding assistant. "
    "Your job is to identify whether two code snippets behave differently in terms of functionality."
)

USER_PROMPT_TEMPLATE = (
    "Here is the original (source) code:\n\n{source}\n\n"
    "And here is the mutated version of the code:\n\n{mutant}\n\n"
    "Do these two pieces of code have any **functional difference**? Respond strictly with \"Yes\" or \"No\" only. "
    "Explain your answer in single line if required provide code responsible with line number in mutant, "
    "if there are multiple discrepancies each will have one line explanation."
)


def read_file_clean(location: str) -> str:
    """
    Reads a C source code file, removes comments, newlines, and tabs,
    and returns compact code suitable for functional comparison.

    Parameters:
        location (str): Path to the C source code file.

    Returns:
        str: Cleaned and compact code.

    Raises:
        FileNotFoundError: If the file does not exist.
        PermissionError: If reading is not allowed.
        IOError: For other I/O issues.
    """
    if not os.path.isfile(location):
        raise FileNotFoundError(f"The file at '{location}' does not exist.")

    try:
        with open(location, 'r', encoding='utf-8') as file:
            code = file.read()

        # Remove block comments (/* ... */)
        code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)

        # Remove line comments (// ...)
        code = re.sub(r'//.*', '', code)

        # Remove tabs, newlines, and multiple spaces
        code = code.replace('\n', ' ').replace('\t', ' ')
        code = re.sub(r'\s+', ' ', code).strip()

        return code

    except PermissionError:
        raise PermissionError(f"Permission denied while reading '{location}'.")
    except IOError as e:
        raise IOError(f"An I/O error occurred while reading '{location}': {e}")


def get_user_prompt(source: str, mutant: str) -> str:
    """Create the user prompt from source and mutant code strings."""
    return USER_PROMPT_TEMPLATE.format(source=source, mutant=mutant)


def get_user_prompt_from_files(source_path: str, mutant_path: str) -> str:
    """Read files and build the user prompt using their cleaned contents."""
    source_code = read_file_clean(source_path)
    mutant_code = read_file_clean(mutant_path)
    return get_user_prompt(source_code, mutant_code)

print("✅ File reading and prompt functions ready!")


## API Interaction Functions

Functions to query Gemini and DeepSeek APIs with proper error handling.


In [ ]:
def compare_code_with_gemini(source_path: str, mutant_path: str) -> Dict[str, Any]:
    """
    Compare code files using Gemini API.
    
    Args:
        source_path: Path to source code file
        mutant_path: Path to mutant code file
        
    Returns:
        Dictionary with response, error, timing, and metadata
    """
    if gemini_model is None:
        return {
            "model": "gemini-2.5-pro",
            "response": None,
            "error": "Gemini API not configured",
            "response_time": 0,
            "source_path": source_path,
            "mutant_path": mutant_path,
            "timestamp": datetime.now().isoformat()
        }
    
    try:
        user_prompt = get_user_prompt_from_files(source_path, mutant_path)
        gemini_input = f"System Prompt: {SYSTEM_PROMPT}\n\nUser Prompt: {user_prompt}"
        
        start_time = time.time()
        resp = gemini_model.generate_content(gemini_input)
        response_time = time.time() - start_time
        
        return {
            "model": "gemini-2.5-pro",
            "response": resp.text,
            "error": None,
            "response_time": response_time,
            "source_path": source_path,
            "mutant_path": mutant_path,
            "timestamp": datetime.now().isoformat()
        }
    except FileNotFoundError as e:
        return {
            "model": "gemini-2.5-pro",
            "response": None,
            "error": f"File not found: {str(e)}",
            "response_time": 0,
            "source_path": source_path,
            "mutant_path": mutant_path,
            "timestamp": datetime.now().isoformat()
        }
    except Exception as e:
        return {
            "model": "gemini-2.5-pro",
            "response": None,
            "error": str(e),
            "response_time": 0,
            "source_path": source_path,
            "mutant_path": mutant_path,
            "timestamp": datetime.now().isoformat()
        }


def compare_code_with_deepseek(source_path: str, mutant_path: str, model_name: str = "deepseek-chat") -> Dict[str, Any]:
    """
    Compare code files using DeepSeek API.
    
    Args:
        source_path: Path to source code file
        mutant_path: Path to mutant code file
        model_name: DeepSeek model to use (default: deepseek-chat)
        
    Returns:
        Dictionary with response, error, timing, and metadata
    """
    if deepseek_headers is None:
        return {
            "model": model_name,
            "response": None,
            "error": "DeepSeek API not configured",
            "response_time": 0,
            "source_path": source_path,
            "mutant_path": mutant_path,
            "timestamp": datetime.now().isoformat()
        }
    
    try:
        user_prompt = get_user_prompt_from_files(source_path, mutant_path)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
        
        payload = {
            "model": model_name,
            "messages": messages,
            "temperature": 0.0,
            "max_tokens": 600
        }
        
        start_time = time.time()
        response = requests.post(
            DEEPSEEK_API_URL,
            headers=deepseek_headers,
            json=payload,
            timeout=60
        )
        response_time = time.time() - start_time
        
        if response.status_code == 200:
            data = response.json()
            content = data["choices"][0]["message"]["content"]
            return {
                "model": model_name,
                "response": content,
                "error": None,
                "response_time": response_time,
                "source_path": source_path,
                "mutant_path": mutant_path,
                "timestamp": datetime.now().isoformat()
            }
        else:
            return {
                "model": model_name,
                "response": None,
                "error": f"HTTP {response.status_code}: {response.text[:200]}",
                "response_time": response_time,
                "source_path": source_path,
                "mutant_path": mutant_path,
                "timestamp": datetime.now().isoformat()
            }
    except FileNotFoundError as e:
        return {
            "model": model_name,
            "response": None,
            "error": f"File not found: {str(e)}",
            "response_time": 0,
            "source_path": source_path,
            "mutant_path": mutant_path,
            "timestamp": datetime.now().isoformat()
        }
    except Exception as e:
        return {
            "model": model_name,
            "response": None,
            "error": str(e),
            "response_time": 0,
            "source_path": source_path,
            "mutant_path": mutant_path,
            "timestamp": datetime.now().isoformat()
        }

print("✅ API interaction functions ready!")


### Option: Secure Input via Jupyter (One-time session)

If you prefer to enter keys interactively (keys are hidden and only stored in memory for the session):


## Batch Processing Function

Process multiple source/mutant pairs and store results in a pandas DataFrame.


In [ ]:
def batch_compare_code_files(
    file_pairs: List[Tuple[str, str]],
    use_gemini: bool = True,
    use_deepseek: bool = True,
    delay_between_requests: float = 1.0
) -> pd.DataFrame:
    """
    Process multiple source/mutant file pairs and store results in a pandas DataFrame.
    
    Args:
        file_pairs: List of tuples (source_path, mutant_path)
        use_gemini: Whether to query Gemini API (default: True)
        use_deepseek: Whether to query DeepSeek API (default: True)
        delay_between_requests: Delay in seconds between API requests to avoid rate limiting (default: 1.0)
        
    Returns:
        pandas DataFrame with columns:
            - source_path: Path to source file
            - mutant_path: Path to mutant file
            - gemini_response: Gemini's response (or None if error/not used)
            - gemini_error: Error message from Gemini (or None)
            - gemini_response_time: Time taken for Gemini response (seconds)
            - deepseek_response: DeepSeek's response (or None if error/not used)
            - deepseek_error: Error message from DeepSeek (or None)
            - deepseek_response_time: Time taken for DeepSeek response (seconds)
            - timestamp: When the comparison was made
    """
    if not file_pairs:
        print("⚠️  No file pairs provided. Returning empty DataFrame.")
        return pd.DataFrame()
    
    if not use_gemini and not use_deepseek:
        print("⚠️  Both APIs disabled. Enable at least one API to run comparisons.")
        return pd.DataFrame()
    
    results = []
    total_pairs = len(file_pairs)
    
    print(f"🚀 Starting batch comparison of {total_pairs} file pair(s)...")
    print(f"   - Gemini: {'Enabled' if use_gemini and gemini_model else 'Disabled'}")
    print(f"   - DeepSeek: {'Enabled' if use_deepseek and deepseek_headers else 'Disabled'}")
    print("=" * 60)
    
    for idx, (source_path, mutant_path) in enumerate(file_pairs, 1):
        print(f"\n[{idx}/{total_pairs}] Processing: {os.path.basename(source_path)} vs {os.path.basename(mutant_path)}")
        
        result_row = {
            'source_path': source_path,
            'mutant_path': mutant_path,
            'gemini_response': None,
            'gemini_error': None,
            'gemini_response_time': 0.0,
            'deepseek_response': None,
            'deepseek_error': None,
            'deepseek_response_time': 0.0,
            'timestamp': datetime.now().isoformat()
        }
        
        # Query Gemini
        if use_gemini and gemini_model:
            print("  🤖 Querying Gemini...", end=" ", flush=True)
            gemini_result = compare_code_with_gemini(source_path, mutant_path)
            result_row['gemini_response'] = gemini_result.get('response')
            result_row['gemini_error'] = gemini_result.get('error')
            result_row['gemini_response_time'] = gemini_result.get('response_time', 0.0)
            
            if gemini_result.get('error'):
                print(f"❌ Error: {gemini_result['error'][:50]}")
            else:
                print(f"✅ ({gemini_result.get('response_time', 0):.2f}s)")
            
            if delay_between_requests > 0:
                time.sleep(delay_between_requests)
        elif use_gemini:
            result_row['gemini_error'] = 'Gemini API not configured'
        
        # Query DeepSeek
        if use_deepseek and deepseek_headers:
            print("  🔍 Querying DeepSeek...", end=" ", flush=True)
            deepseek_result = compare_code_with_deepseek(source_path, mutant_path)
            result_row['deepseek_response'] = deepseek_result.get('response')
            result_row['deepseek_error'] = deepseek_result.get('error')
            result_row['deepseek_response_time'] = deepseek_result.get('response_time', 0.0)
            
            if deepseek_result.get('error'):
                print(f"❌ Error: {deepseek_result['error'][:50]}")
            else:
                print(f"✅ ({deepseek_result.get('response_time', 0):.2f}s)")
            
            if delay_between_requests > 0:
                time.sleep(delay_between_requests)
        elif use_deepseek:
            result_row['deepseek_error'] = 'DeepSeek API not configured'
        
        results.append(result_row)
    
    print("\n" + "=" * 60)
    print(f"✅ Batch processing complete! Processed {len(results)} pair(s).")
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    # Add summary statistics
    if len(df) > 0:
        print("\n📊 Summary Statistics:")
        if 'gemini_error' in df.columns:
            gemini_success = df['gemini_error'].isna().sum()
            print(f"   Gemini: {gemini_success}/{len(df)} successful")
        if 'deepseek_error' in df.columns:
            deepseek_success = df['deepseek_error'].isna().sum()
            print(f"   DeepSeek: {deepseek_success}/{len(df)} successful")
        
        if 'gemini_response_time' in df.columns:
            avg_gemini_time = df['gemini_response_time'].mean()
            print(f"   Avg Gemini time: {avg_gemini_time:.2f}s")
        if 'deepseek_response_time' in df.columns:
            avg_deepseek_time = df['deepseek_response_time'].mean()
            print(f"   Avg DeepSeek time: {avg_deepseek_time:.2f}s")
    
    return df

print("✅ Batch processing function ready!")


## Utility Functions for File Pair Generation

Functions to automatically collect .c files and generate source/mutant pairs from the source folders.


In [ ]:
def collect_c_files(base_dir: str = ".") -> List[str]:
    """
    Collect all .c files from source/NTS and source/tcas folders.
    
    Args:
        base_dir: Base directory path (default: current directory)
        
    Returns:
        List of absolute paths to all .c files found in source/NTS and source/tcas folders
    """
    folders = [
        os.path.join(base_dir, "source", "NTS"),
        os.path.join(base_dir, "source", "tcas")
    ]
    
    c_files = []
    
    for folder in folders:
        if not os.path.isdir(folder):
            print(f"⚠️  Folder not found: {folder}")
            continue
        
        print(f"📁 Scanning folder: {folder}")
        
        # Walk through all subdirectories
        for root, dirs, files in os.walk(folder):
            for file in files:
                if file.endswith('.c'):
                    file_path = os.path.abspath(os.path.join(root, file))
                    c_files.append(file_path)
                    print(f"   ✅ Found: {file_path}")
    
    print(f"\n📊 Total .c files found: {len(c_files)}")
    return sorted(c_files)


def create_file_pairs_from_folders(base_dir: str = ".") -> List[Tuple[str, str]]:
    """
    Create source/mutant file pairs from source/NTS and source/tcas folders recursively.
    
    This function handles nested directory structures where:
    - Source files are typically at a project/group level (e.g., merge2BSTree/merge_2_bst_source.c)
    - Mutant files are in subdirectories (e.g., merge2BSTree/v1/merge_2_bst.c, v2/, v3/, etc.)
    
    The function:
    1. Recursively walks through all subdirectories
    2. Groups files by their project directory (immediate subdirectory of NTS/tcas)
    3. Identifies source files (files with "_source.c" in name at project root)
    4. Pairs each source with all mutants found in subdirectories of that project
    
    Args:
        base_dir: Base directory path (default: current directory)
        
    Returns:
        List of tuples (source_path, mutant_path) representing file pairs
    """
    folders = [
        os.path.join(base_dir, "source", "NTS"),
        os.path.join(base_dir, "source", "tcas")
    ]
    
    file_pairs = []
    
    for folder in folders:
        if not os.path.isdir(folder):
            print(f"⚠️  Folder not found: {folder}")
            continue
        
        print(f"\n📁 Processing folder: {folder}")
        
        # Step 1: Identify project directories (direct subdirectories of NTS/tcas)
        project_dirs = []
        files_in_root = []  # Files directly in the folder (for cases like tcas)
        
        try:
            for item in os.listdir(folder):
                item_path = os.path.join(folder, item)
                if os.path.isdir(item_path):
                    project_dirs.append(item_path)
                elif item.endswith('.c'):
                    files_in_root.append(os.path.abspath(item_path))
        except Exception as e:
            print(f"   ⚠️  Error reading folder: {e}")
            continue
        
        # Step 2a: Handle files directly in folder (e.g., tcas folder structure)
        if files_in_root:
            # Files are in the root folder - treat as a single project
            source_files = []
            mutant_files = []
            
            for file_path in files_in_root:
                file_name = os.path.basename(file_path)
                
                # Identify source files: files with "_source.c" in name
                if "_source.c" in file_name.lower():
                    source_files.append(file_path)
                else:
                    mutant_files.append(file_path)
            
            # Handle cases where no explicit source file found
            if not source_files and mutant_files:
                # Look for files that might be source (e.g., "source.c")
                for file_path in mutant_files:
                    file_name = os.path.basename(file_path).lower()
                    if "source" in file_name and file_name.endswith('.c'):
                        source_files.append(file_path)
                        mutant_files.remove(file_path)
                        break
                
                # If still no source, use first file as source
                if not source_files and mutant_files:
                    source_files.append(mutant_files[0])
                    mutant_files.remove(mutant_files[0])
            
            # Create pairs for root-level files
            if source_files and mutant_files:
                folder_name = os.path.basename(folder)
                print(f"\n   📂 Project: {folder_name} (root level)")
                for source_file in source_files:
                    for mutant_file in mutant_files:
                        file_pairs.append((source_file, mutant_file))
                        source_name = os.path.basename(source_file)
                        mutant_name = os.path.basename(mutant_file)
                        print(f"      ✅ Pair: {source_name} -> {mutant_name}")
            elif source_files and not mutant_files:
                print(f"\n   ⚠️  No mutants found for source in {os.path.basename(folder)}")
            elif not source_files and mutant_files:
                print(f"\n   ⚠️  No source file found in {os.path.basename(folder)}, {len(mutant_files)} mutant(s) skipped")
        
        # Step 2b: For each project subdirectory, collect source and mutant files
        for project_dir in project_dirs:
            project_name = os.path.basename(project_dir)
            print(f"\n   📂 Project: {project_name}")
            
            source_files = []
            mutant_files = []
            
            # Walk through project directory recursively
            for root, dirs, files in os.walk(project_dir):
                c_files = [f for f in files if f.endswith('.c')]
                
                for c_file in c_files:
                    file_path = os.path.abspath(os.path.join(root, c_file))
                    file_name = os.path.basename(file_path)
                    
                    # Check if file is at project root level
                    is_at_project_root = (os.path.dirname(file_path) == project_dir)
                    
                    # Identify source files: files with "_source.c" at project root
                    if "_source.c" in file_name.lower() and is_at_project_root:
                        source_files.append(file_path)
                    else:
                        # All other files are considered mutants
                        mutant_files.append(file_path)
            
            # Step 3: Handle cases where no explicit source file found
            if not source_files and mutant_files:
                # Look for files at project root that could be source
                project_root_files = [f for f in mutant_files if os.path.dirname(f) == project_dir]
                if project_root_files:
                    # Use first file at project root as source
                    source_files.append(project_root_files[0])
                    mutant_files.remove(project_root_files[0])
                elif mutant_files:
                    # Use first mutant as source if nothing else
                    source_files.append(mutant_files[0])
                    mutant_files.remove(mutant_files[0])
            
            # Step 4: Create pairs
            if source_files and mutant_files:
                for source_file in source_files:
                    for mutant_file in mutant_files:
                        file_pairs.append((source_file, mutant_file))
                        source_name = os.path.basename(source_file)
                        mutant_rel = os.path.relpath(mutant_file, project_dir)
                        print(f"      ✅ Pair: {source_name} -> {mutant_rel}")
            elif source_files and not mutant_files:
                print(f"      ⚠️  No mutants found for source: {os.path.basename(source_files[0])}")
            elif not source_files and mutant_files:
                print(f"      ⚠️  No source file found, {len(mutant_files)} mutant(s) skipped")
    
    print(f"\n📊 Total file pairs created: {len(file_pairs)}")
    return file_pairs

print("✅ Utility functions ready!")


### Example Usage

Use these functions to automatically generate file pairs from your source folders.


In [ ]:
# Example 1: Collect all .c files
all_c_files = collect_c_files(base_dir=".")
print(f"\nFound {len(all_c_files)} .c files")

# Example 2: Automatically create file pairs from folders
file_pairs = create_file_pairs_from_folders(base_dir=".")
print(f"\nCreated {len(file_pairs)} file pairs")

# Example 3: Use the file pairs for batch comparison
# results_df = batch_compare_code_files(
#     file_pairs=file_pairs,
#     use_gemini=True,
#     use_deepseek=True,
#     delay_between_requests=1.0
# )


In [ ]:
s = set([file[0] for file in file_pairs])
s

## Example Usage

Run batch comparisons with your source and mutant file pairs.


In [ ]:
# Run batch comparison
# Set use_gemini=False or use_deepseek=False to disable specific APIs
results_df = batch_compare_code_files(
    file_pairs=file_pairs,
    use_gemini=True,
    use_deepseek=True,
    delay_between_requests=1.0  # Delay to avoid rate limiting
)

# Display results
print("\n📋 Results DataFrame:")
print(results_df)

# Save to CSV if needed
# results_df.to_csv('comparison_results.csv', index=False)


In [ ]:
results_df.to_csv('comparison_results.csv', index=False)

## Analyze Results

Helper functions to analyze and view the comparison results.


In [ ]:
def view_results_summary(df: pd.DataFrame):
    """Display a summary of the results DataFrame."""
    if df.empty:
        print("No results to display.")
        return
    
    print(f"\n📊 Results Summary:")
    print(f"   Total comparisons: {len(df)}")
    
    # Gemini stats
    if 'gemini_response' in df.columns:
        gemini_success = df['gemini_response'].notna().sum()
        gemini_errors = df['gemini_error'].notna().sum()
        print(f"\n   Gemini:")
        print(f"      Successful: {gemini_success}")
        print(f"      Errors: {gemini_errors}")
        if gemini_success > 0:
            avg_time = df[df['gemini_response'].notna()]['gemini_response_time'].mean()
            print(f"      Avg response time: {avg_time:.2f}s")
    
    # DeepSeek stats
    if 'deepseek_response' in df.columns:
        deepseek_success = df['deepseek_response'].notna().sum()
        deepseek_errors = df['deepseek_error'].notna().sum()
        print(f"\n   DeepSeek:")
        print(f"      Successful: {deepseek_success}")
        print(f"      Errors: {deepseek_errors}")
        if deepseek_success > 0:
            avg_time = df[df['deepseek_response'].notna()]['deepseek_response_time'].mean()
            print(f"      Avg response time: {avg_time:.2f}s")
    
    # Show errors if any
    if 'gemini_error' in df.columns:
        gemini_errors_df = df[df['gemini_error'].notna()]
        if not gemini_errors_df.empty:
            print(f"\n   Gemini Errors:")
            for idx, row in gemini_errors_df.iterrows():
                print(f"      {row['source_path']}: {row['gemini_error']}")
    
    if 'deepseek_error' in df.columns:
        deepseek_errors_df = df[df['deepseek_error'].notna()]
        if not deepseek_errors_df.empty:
            print(f"\n   DeepSeek Errors:")
            for idx, row in deepseek_errors_df.iterrows():
                print(f"      {row['source_path']}: {row['deepseek_error']}")


def compare_responses(df: pd.DataFrame, row_index: int = 0):
    """Compare Gemini and DeepSeek responses for a specific row."""
    if df.empty:
        print("No results to display.")
        return
    
    if row_index >= len(df):
        print(f"Row index {row_index} out of range. DataFrame has {len(df)} rows.")
        return
    
    row = df.iloc[row_index]
    print(f"\n📝 Comparison for row {row_index}:")
    print(f"   Source: {row['source_path']}")
    print(f"   Mutant: {row['mutant_path']}")
    print("\n" + "=" * 60)
    
    if row['gemini_response']:
        print("\n🤖 Gemini Response:")
        print(row['gemini_response'])
        print(f"\n   Response time: {row['gemini_response_time']:.2f}s")
    elif row['gemini_error']:
        print("\n🤖 Gemini Error:")
        print(row['gemini_error'])
    
    if row['deepseek_response']:
        print("\n🔍 DeepSeek Response:")
        print(row['deepseek_response'])
        print(f"\n   Response time: {row['deepseek_response_time']:.2f}s")
    elif row['deepseek_error']:
        print("\n🔍 DeepSeek Error:")
        print(row['deepseek_error'])
    
    print("\n" + "=" * 60)

print("✅ Analysis functions ready!")


In [ ]:
results_df